# 序号28：策略研究报告

## 阶段3 综合作战项目

### 任务
选取 1-2 个策略，完成全流程：**策略逻辑 → 参数优化 → 回测验证 → 风险控制 → 绩效归因 → 研究报告**

### 验收标准
- ✅ 有样本内外划分
- ✅ 考虑手续费和滑点
- ✅ 对比基准（买入持有）
- ✅ 讨论局限性和适用条件
- ✅ 上传 GitHub

### 本报告策略
**动量策略**（Momentum Strategy）：基于 Jegadeesh-Titman (1993)，买入过去表现强的股票，卖出过去表现弱的股票。
在沪深300成分股中，每月选择过去12个月（跳过最近1个月）收益最高的20只股票等权持有，月度调仓。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import akshare as ak
import warnings, time
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('✅ 环境就绪')

---

## 1. 策略逻辑与假设

### 1.1 投资逻辑

**动量效应**（Momentum Effect）是学术界最稳健的异象之一：
- Jegadeesh & Titman (1993) 发现，过去 3-12 个月表现好的股票，在未来 3-12 个月继续表现好
- 行为金融解释：反应不足（投资者对新信息消化缓慢）、处置效应、羊群效应强化

### 1.2 策略规则

| 参数 | 取值 | 说明 |
|------|------|------|
| 股票池 | 沪深300成分股 | 流动性好、代表性强 |
| 动量计算期 | 12个月（跳过最近1月） | 标准 Jegadeesh-Titman 设定 |
| 持有期 | 1个月 | 月度调仓 |
| 持仓数量 | Top 20 | 等权配置 |
| 交易成本 | 双边 0.15% | 含佣金+印花税+滑点 |
| 基准 | 沪深300等权买入持有 | 被动策略对比 |

---

## 2. 数据与方法

### 2.1 数据获取

回测期间：2021-01 至 2026-06，样本内 2021-2024，样本外 2025-2026。
获取沪深300成分股月线数据。

In [ ]:
# 获取沪深300成分股
print('📡 Step 1: 获取沪深300成分股...')
hs300 = ak.index_stock_cons_csindex(symbol='000300')
all_stocks = hs300['成分券代码'].tolist()
# 取前40只大市值股票（控制API调用量）
sample_stocks = all_stocks[:40]
print(f'沪深300成分股: {len(all_stocks)}只 → 演示用前{len(sample_stocks)}只')

# 获取月线数据（更稳定，覆盖更长历史）
print('\n📡 Step 2: 获取月线数据 (预计1-2分钟)...')
monthly_price = {}
for i, code in enumerate(sample_stocks):
    try:
        df = ak.stock_zh_a_hist(symbol=code, period='monthly',
                                start_date='20190101', end_date='20260624',
                                adjust='qfq')
        df['日期'] = pd.to_datetime(df['日期'])
        df = df.set_index('日期').sort_index()
        if len(df) >= 24:  # 至少2年数据
            monthly_price[code] = df['收盘']
    except: pass
    if (i+1) % 10 == 0: print(f'  进度: {i+1}/{len(sample_stocks)}')
    time.sleep(0.2)

price_m = pd.DataFrame(monthly_price)
ret_m = price_m.pct_change().dropna()

print(f'\n月线数据: {price_m.shape}')
print(f'日期: {price_m.index[0].date()} ~ {price_m.index[-1].date()}')
print(f'可用股票数: {len(price_m.columns)}')

### 2.2 样本内外划分

严格的时间序列划分：
- **样本内（训练）**：2021-01 ~ 2024-12（48个月）
- **样本外（测试）**：2025-01 ~ 2026-06（18个月）

> ⚠️ 样本外数据在整个策略开发过程中**绝对不碰**，只在最终验证时使用一次。

In [ ]:
# 划分样本
in_sample_end = '2024-12-31'
out_sample_start = '2025-01-01'

price_is = price_m[price_m.index <= in_sample_end]
price_os = price_m[price_m.index > in_sample_end]

print(f'样本内: {price_is.index[0].date()} ~ {price_is.index[-1].date()}, {len(price_is)}个月')
print(f'样本外: {price_os.index[0].date()} ~ {price_os.index[-1].date()}, {len(price_os)}个月')

---

## 3. 回测框架

### 3.1 回测引擎

实现月度调仓的动量策略回测函数：
1. 每月末计算过去12个月（跳过最近1月）的动量排名
2. 选择 Top 20 等权持有
3. 计算换手率和交易成本
4. 记录组合收益

In [ ]:
def backtest_momentum(price_df, top_n=20, lookback=12, skip=1, cost_rate=0.0015):
    """
    动量策略回测引擎
    
    参数:
        price_df: 月度价格 DataFrame
        top_n: 持仓股票数量
        lookback: 动量计算期（月）
        skip: 跳过的最近月份
        cost_rate: 单边交易成本率
    
    返回:
        results: dict，含组合收益、换手率、各期持仓等
    """
    ret_df = price_df.pct_change()
    n_months = len(price_df)
    
    portfolio_returns = []
    turnovers = []
    holdings_history = []  # 每期持仓
    prev_holdings = set()
    
    # 需要足够的历史数据
    start_idx = lookback + skip
    
    for t in range(start_idx, n_months - 1):
        # 计算动量：过去lookback个月（跳过skip个月）的累计收益
        momentum_start = t - lookback - skip
        momentum_end = t - skip
        
        # 过去lookback个月的收益
        past_ret = ret_df.iloc[momentum_start:momentum_end]
        momentum = (1 + past_ret).prod() - 1
        momentum = momentum.dropna()
        
        if len(momentum) < top_n:
            # 股票不够，跳过
            portfolio_returns.append(np.nan)
            turnovers.append(np.nan)
            holdings_history.append(set())
            continue
        
        # 选Top N
        top_stocks = momentum.nlargest(top_n).index.tolist()
        cur_holdings = set(top_stocks)
        holdings_history.append(cur_holdings)
        
        # 计算换手率
        if prev_holdings:
            sold = prev_holdings - cur_holdings
            bought = cur_holdings - prev_holdings
            turnover = (len(sold) + len(bought)) / (2 * top_n)
        else:
            turnover = 1.0  # 初始建仓全换手
        turnovers.append(turnover)
        prev_holdings = cur_holdings
        
        # 下一期收益（等权）
        next_ret = ret_df.iloc[t + 1][top_stocks].mean()
        
        # 扣除交易成本
        cost = turnover * cost_rate * 2  # 双边成本
        net_ret = next_ret - cost
        portfolio_returns.append(net_ret)
    
    # 对齐日期
    dates = ret_df.index[start_idx + 1:start_idx + 1 + len(portfolio_returns)]
    
    results = {
        'returns': pd.Series(portfolio_returns, index=dates),
        'turnovers': pd.Series(turnovers, index=dates),
        'holdings': holdings_history,
        'dates': dates
    }
    return results

print('✅ 回测引擎就绪')

### 3.2 样本内回测

In [ ]:
# 样本内回测
print('🔄 样本内回测运行中...')
res_is = backtest_momentum(price_is, top_n=20, lookback=12, skip=1, cost_rate=0.0015)

ret_is = res_is['returns'].dropna()
print(f'样本内回测: {len(ret_is)} 个月')
print(f'累计收益: {ret_is.add(1).prod() - 1:.2%}')
print(f'年化收益: {ret_is.add(1).prod() ** (12/len(ret_is)) - 1:.2%}')
print(f'年化波动: {ret_is.std() * np.sqrt(12):.2%}')
print(f'年化夏普: {ret_is.mean() / ret_is.std() * np.sqrt(12):.2f}')
print(f'最大回撤: {(ret_is.add(1).cumprod() / ret_is.add(1).cumprod().cummax() - 1).min():.2%}')
print(f'平均换手率: {res_is["turnovers"].mean():.1%}')

In [ ]:
# 基准：等权买入持有
bench_is = price_is.pct_change().dropna().mean(axis=1)
bench_is = bench_is[bench_is.index.isin(ret_is.index)]

cum_is = ret_is.add(1).cumprod()
cum_bh_is = bench_is.add(1).cumprod()

print(f'===== 样本内：策略 vs 基准 =====')
print(f'策略累计: {cum_is.iloc[-1] - 1:.2%}')
print(f'基准累计: {cum_bh_is.iloc[-1] - 1:.2%}')
print(f'超额收益: {(cum_is.iloc[-1] - cum_bh_is.iloc[-1]):.2%}')

excess_is = ret_is - bench_is
print(f'年化超额: {excess_is.mean() * 12:.2%}')
print(f'信息比率: {excess_is.mean() / excess_is.std() * np.sqrt(12):.2f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 累计收益
ax = axes[0,0]
ax.plot(cum_is.index, cum_is.values, color='#2563eb', linewidth=2, label='动量策略')
ax.plot(cum_bh_is.index, cum_bh_is.values, color='gray', linewidth=1.5, alpha=0.7, label='等权买入持有')
ax.fill_between(cum_is.index, cum_is.values, cum_bh_is.values, alpha=0.1, color='green')
ax.set_title('样本内累计收益', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

# 2. 回撤
ax = axes[0,1]
dd_is = cum_is / cum_is.cummax() - 1
dd_bh = cum_bh_is / cum_bh_is.cummax() - 1
ax.fill_between(dd_is.index, 0, dd_is.values, alpha=0.3, color='#dc2626', label='策略回撤')
ax.fill_between(dd_bh.index, 0, dd_bh.values, alpha=0.2, color='gray', label='基准回撤')
ax.set_title('回撤曲线', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

# 3. 月度收益分布
ax = axes[1,0]
ax.hist(ret_is * 100, bins=25, color='steelblue', alpha=0.8, edgecolor='white', label='策略月收益')
ax.hist(bench_is * 100, bins=25, color='gray', alpha=0.4, edgecolor='white', label='基准月收益')
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('月度收益 (%)')
ax.set_title('月收益分布', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

# 4. 滚动12月夏普
ax = axes[1,1]
roll_sharpe_is = ret_is.rolling(12).apply(lambda x: x.mean()/x.std()*np.sqrt(12) if x.std()>0 else 0)
roll_sharpe_bh = bench_is.rolling(12).apply(lambda x: x.mean()/x.std()*np.sqrt(12) if x.std()>0 else 0)
ax.plot(roll_sharpe_is.index, roll_sharpe_is.values, color='#2563eb', linewidth=1.5, label='策略滚动夏普')
ax.plot(roll_sharpe_bh.index, roll_sharpe_bh.values, color='gray', linewidth=1, alpha=0.7, label='基准滚动夏普')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_title('滚动12月夏普比率', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. 样本外验证

用完全相同的策略参数在样本外数据上运行——**这是检验策略是否有效的唯一标准。**

In [ ]:
# 样本外回测（完全相同的参数，绝不调参）
print('🔄 样本外回测运行中...')
res_os = backtest_momentum(price_os, top_n=20, lookback=12, skip=1, cost_rate=0.0015)

ret_os = res_os['returns'].dropna()
bench_os = price_os.pct_change().dropna().mean(axis=1)
bench_os = bench_os[bench_os.index.isin(ret_os.index)]

cum_os = ret_os.add(1).cumprod()
cum_bh_os = bench_os.add(1).cumprod()

print(f'===== 样本外 =====')
print(f'策略累计: {cum_os.iloc[-1] - 1:.2%}')
print(f'基准累计: {cum_bh_os.iloc[-1] - 1:.2%}')
print(f'超额:     {(cum_os.iloc[-1] - cum_bh_os.iloc[-1]):.2%}')
print(f'年化夏普: {ret_os.mean() / ret_os.std() * np.sqrt(12):.2f}')
print(f'最大回撤: {(cum_os / cum_os.cummax() - 1).min():.2%}')

In [ ]:
# 样本内外对比可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 左：样本外累计收益
ax = axes[0]
ax.plot(cum_os.index, cum_os.values, color='#2563eb', linewidth=2, label='动量策略')
ax.plot(cum_bh_os.index, cum_bh_os.values, color='gray', linewidth=1.5, alpha=0.7, label='基准')
ax.set_title('样本外累计收益', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

# 中：样本内外指标对比
ax = axes[1]
metrics = ['年化收益', '年化夏普', '最大回撤', '胜率']
is_vals = [
    ret_is.add(1).prod()**(12/len(ret_is))-1,
    ret_is.mean()/ret_is.std()*np.sqrt(12),
    (cum_is/cum_is.cummax()-1).min(),
    (ret_is>0).mean()
]
os_vals = [
    ret_os.add(1).prod()**(12/len(ret_os))-1,
    ret_os.mean()/ret_os.std()*np.sqrt(12),
    (cum_os/cum_os.cummax()-1).min(),
    (ret_os>0).mean()
]
x = np.arange(len(metrics))
w = 0.35
ax.bar(x-w/2, [v*100 for v in is_vals], w, color='#2563eb', alpha=0.8, label='样本内')
ax.bar(x+w/2, [v*100 if v is not None else 0 for v in os_vals], w, color='#f59e0b', alpha=0.8, label='样本外')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=9)
ax.set_title('样本内外指标对比', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# 右：全周期合并
ax = axes[2]
ret_all = pd.concat([ret_is, ret_os])
bench_all = pd.concat([bench_is, bench_os])
cum_all = ret_all.add(1).cumprod()
cum_bh_all = bench_all.add(1).cumprod()

ax.plot(cum_all.index, cum_all.values, color='#2563eb', linewidth=2, label='动量策略')
ax.plot(cum_bh_all.index, cum_bh_all.values, color='gray', linewidth=1.5, alpha=0.7, label='基准')
ax.axvline(x=pd.Timestamp('2025-01-01'), color='red', linestyle='--', alpha=0.5, label='样本外开始')
ax.set_title('全周期累计收益', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. 风险分析

In [ ]:
# 风险指标
ret_all = pd.concat([ret_is, ret_os])
cum_all = ret_all.add(1).cumprod()

print('='*50)
print('        风 险 分 析 报 告')
print('='*50)

# 最大回撤
dd_all = cum_all / cum_all.cummax() - 1
max_dd = dd_all.min()
max_dd_date = dd_all.idxmin()

# VaR & CVaR
var_95 = np.percentile(ret_all, 5)
cvar_95 = ret_all[ret_all <= var_95].mean()
var_99 = np.percentile(ret_all, 1)
cvar_99 = ret_all[ret_all <= var_99].mean()

# 偏度 & 峰度
skewness = ret_all.skew()
kurtosis = ret_all.kurtosis()

# 最大连续亏损月数
neg_streak = (ret_all < 0).astype(int)
from itertools import groupby
max_losing_streak = max((sum(1 for _ in g) for k, g in groupby(neg_streak) if k == 1), default=0)

print(f'最大回撤:        {max_dd:.2%} (日期: {max_dd_date.date()})')
print(f'月VaR(95%):     {var_95:.2%}')
print(f'月CVaR(95%):    {cvar_95:.2%}')
print(f'月VaR(99%):     {var_99:.2%}')
print(f'月CVaR(99%):    {cvar_99:.2%}')
print(f'偏度:            {skewness:.3f}', '(负偏=暴跌风险大)' if skewness < -0.5 else '')
print(f'超额峰度:        {kurtosis:.3f}', '(>0=肥尾风险)' if kurtosis > 0 else '')
print(f'最长连续亏损:    {max_losing_streak} 个月')
print(f'月度胜率:        {(ret_all>0).mean():.1%}')
print(f'盈亏比:          {ret_all[ret_all>0].mean() / abs(ret_all[ret_all<0].mean()):.2f}')
print(f'Calmar比率:      {ret_all.mean()*12 / abs(max_dd):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 回撤全景
ax = axes[0]
ax.fill_between(dd_all.index, 0, dd_all.values*100, color='#dc2626', alpha=0.3)
ax.plot(dd_all.index, dd_all.values*100, color='#dc2626', linewidth=1)
ax.set_title(f'回撤全景 (最大: {max_dd:.1%})', fontsize=12, fontweight='bold')
ax.set_ylabel('回撤 (%)')
ax.grid(True, alpha=0.3)

# 收益分布 + VaR
ax = axes[1]
ax.hist(ret_all*100, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(x=var_95*100, color='red', linestyle='--', linewidth=1.5, label=f'VaR 95%: {var_95:.1%}')
ax.axvline(x=var_99*100, color='darkred', linestyle='--', linewidth=1.5, label=f'VaR 99%: {var_99:.1%}')
ax.set_xlabel('月度收益 (%)')
ax.set_title('收益分布 & VaR', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# 月度收益热力图
ax = axes[2]
ret_monthly = ret_all.copy()
ret_monthly.index = pd.to_datetime(ret_monthly.index)
ret_matrix = ret_monthly.groupby([ret_monthly.index.year, ret_monthly.index.month]).mean().unstack()
ret_matrix.index = ret_matrix.index.astype(int)
ret_matrix.columns = ['1月','2月','3月','4月','5月','6月','7月','8月','9月','10月','11月','12月'][:len(ret_matrix.columns)]
ret_matrix = ret_matrix[[c for c in ret_matrix.columns if c in ['1月','2月','3月','4月','5月','6月','7月','8月','9月','10月','11月','12月']]]
sns.heatmap(ret_matrix*100, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax,
            cbar_kws={'label': '月收益 (%)'}, linewidths=0.5)
ax.set_title('月度收益热力图', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 6. 绩效归因

用 Fama-French 五因子模型拆解超额收益来源。

In [ ]:
# 因子归因
try:
    print('📡 获取 Fama-French 五因子数据...')
    ff5 = ak.fama_french_5factor_daily(start_date='20210101', end_date='20260624')
    
    # 处理因子数据
    if '日期' in ff5.columns or 'trade_date' in ff5.columns:
        date_col = '日期' if '日期' in ff5.columns else 'trade_date'
        ff5[date_col] = pd.to_datetime(ff5[date_col])
        ff5 = ff5.set_index(date_col).sort_index()
    
    # 月度化因子（累计）
    ff5_m = ff5.resample('ME').apply(lambda x: (1+x).prod()-1)
    
    # 标准化列名
    col_map = {}
    for c in ff5_m.columns:
        cl = c.lower()
        if 'mkt' in cl or 'market' in cl: col_map[c] = 'MKT'
        elif 'smb' in cl: col_map[c] = 'SMB'
        elif 'hml' in cl: col_map[c] = 'HML'
        elif 'rmw' in cl: col_map[c] = 'RMW'
        elif 'cma' in cl: col_map[c] = 'CMA'
        elif 'rf' in cl: col_map[c] = 'RF'
    ff5_m = ff5_m.rename(columns=col_map)
    
    factor_cols = [c for c in ['MKT','SMB','HML','RMW','CMA'] if c in ff5_m.columns]
    print(f'可用因子: {factor_cols}')
    
    # 对月度超额收益做回归
    common = ret_all.index.intersection(ff5_m.index)
    y = ret_all.loc[common] * 100
    X = ff5_m.loc[common, factor_cols]
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    print('\n===== Fama-French 因子归因 =====')
    print(model.summary().tables[1])
    
    alpha_m = model.params.get('const', 0)
    alpha_annual = alpha_m * 12
    alpha_t = model.tvalues.get('const', 0)
    alpha_p = model.pvalues.get('const', 0)
    
    print(f'\n年化 Alpha: {alpha_annual:.2f}%')
    print(f't 统计量: {alpha_t:.2f}')
    print(f'p 值: {alpha_p:.4f}')
    print(f'R²: {model.rsquared:.4f}')
    
    if alpha_p < 0.1 and alpha_annual > 0:
        print('✅ 存在正的选股 Alpha（统计显著）')
    else:
        print('⚠️ Alpha 不显著 → 超额收益主要来自因子暴露')
except Exception as e:
    print(f'因子归因失败: {e}')
    print('跳过因子归因部分')

---

## 7. 局限性与适用条件

### 7.1 本策略的局限性

| 局限 | 说明 | 影响 |
|------|------|------|
| **动量崩溃风险** | 市场急转弯时（如2009年反弹），动量策略往往大幅回撤 | 极端行情下可能亏 20-40% |
| **容量限制** | 小市值股票流动性不足，大资金无法复制 | 策略容量约 1-5 亿 |
| **因子拥挤** | 越来越多的动量基金可能挤压缩超额收益 | 未来 Alpha 可能衰减 |
| **参数敏感性** | 动量计算期、持仓数量等参数选择影响显著 | 需要定期审视参数 |
| **数据局限** | 仅用沪深300前40只股票，代表性有限 | 结果仅作参考，不可直接实盘 |
| **未考虑停牌/涨跌停** | 实际交易中可能无法买入/卖出 | 实际操作难度增大 |

### 7.2 适用条件

| 条件 | 判断 |
|------|------|
| 市场趋势明显（牛市/熊市） | ✅ 动量策略在趋势市中表现好 |
| 震荡盘整市 | ⚠️ 频繁换手 + 反复打脸 |
| 市场风格剧烈切换 | ❌ 动量崩溃风险高 |
| 流动性紧缩期 | ❌ 成本上升 + 流动性差 |

---

## 8. 结论

### 核心发现

1. 动量策略在 A 股沪深300上展现了超额收益潜力
2. 样本外验证是区分运气和能力的唯一标准
3. 交易成本会侵蚀约 1-2% 的年化收益，不可忽略
4. 因子归因揭示超额收益来源——是 Alpha 还是 Beta 暴露

### 改进方向

- 结合估值因子做「价值动量」：只买便宜的动量股，规避动量崩溃
- 加入止损机制：单月回撤超过 X% 减仓
- 扩展股票池：中证500 + 中证1000
- 机器学习增强：用 XGBoost 预测动量持续性

### 验收自检

- [x] 有样本内外划分 ✅
- [x] 考虑手续费和滑点 ✅
- [x] 对比基准（买入持有） ✅
- [x] 讨论局限性和适用条件 ✅
- [x] 上传 GitHub ✅

> **"纸上得来终觉浅，绝知此事要躬行。"** —— 陆游
> 
> 本报告是阶段3的收官之作，整合了策略开发（22）→ 回测框架（24）→ 回测陷阱（26）→ 绩效归因（27）的全流程技能。下一步进入阶段4：机器学习量化。